In [2]:
from eventflow.data import load_train, make_train_examples
from eventflow.validation import add_folds, split
from eventflow.config import EVENT_CLASSES, TUNING_FOLD
from eventflow.features import make_features, CATEGORICAL

train = load_train()
examples = add_folds(make_train_examples(train))
X = make_features(train)
print("feature table:", X.shape)        # expect (1203963, 63)
X.iloc[95:100, :]

feature table: (1203963, 79)


,ev_lag0,sev_lag0,ev_lag1,sev_lag1,ev_lag2,sev_lag2,run_length,cnt5_COMPONENT_REPLACED,cnt5_CONTROL_ALERT,cnt5_COOLING_ACTIVATED,...,share_LOAD_INCREASE,share_MAINTENANCE_CHECK,share_PRESSURE_WARNING,share_RECOVERY,share_ROUTINE_CHECK,share_SENSOR_RESET,share_TEMP_WARNING,share_VIBRATION_WARNING,mean_severity_so_far,events_so_far
95,7,0.0,3,2.0,2,0.0,1,0,0,1,...,0.041667,0.145833,0.031250,0.208333,0.020833,0.010417,0.125000,0.197917,1.072917,96
96,2,0.0,7,0.0,3,2.0,1,0,0,2,...,0.041237,0.144330,0.030928,0.206186,0.020619,0.010309,0.123711,0.195876,1.061856,97
97,2,0.0,2,0.0,7,0.0,2,0,0,3,...,0.040816,0.142857,0.030612,0.204082,0.020408,0.010204,0.122449,0.193878,1.051020,98
98,9,1.0,2,0.0,2,0.0,1,0,0,2,...,0.040404,0.141414,0.030303,0.212121,0.020202,0.010101,0.121212,0.191919,1.050505,99
99,12,1.0,9,1.0,2,0.0,1,0,0,2,...,0.040000,0.140000,0.030000,0.210000,0.020000,0.010000,0.130000,0.190000,1.050000,100


In [3]:
row = int(examples["last_row"].iloc[1234])
seq_id = train.loc[row, "sequence_id"]
seq_start = train.index[train["sequence_id"] == seq_id][0]
history = train.loc[seq_start:row, "event_type"].tolist()

by_hand = {
    "ev_lag0": EVENT_CLASSES.index(history[-1]),
    "ev_lag2": EVENT_CLASSES.index(history[-3]),
    "cnt30_TEMP_WARNING": history[-30:].count("TEMP_WARNING"),
    "cnt5_RECOVERY": history[-5:].count("RECOVERY"),
}
run = 1
while run < len(history) and history[-1 - run] == history[-1]:
    run += 1
by_hand["run_length"] = run

for name, value in by_hand.items():
    assert X.loc[row, name] == value, (name, X.loc[row, name], value)
print("Hand check passed:", by_hand)

Hand check passed: {'ev_lag0': 2, 'ev_lag2': 12, 'cnt30_TEMP_WARNING': 1, 'cnt5_RECOVERY': 2, 'run_length': 2}


In [4]:
# NO-FUTURE TEST (leakage check)
# e.g. a diary of 250 events, cut at row 125:
#   full = make_features(rows 0..249) → features at row 125
#   cut  = make_features(rows 0..125) → features at row 125
#   same      → no feature used rows 126..249 → no leakage 
#   different → some feature peeked at the future → leakage 
lengths = train.groupby("sequence_id").size()
seq_id = lengths.idxmax()                                  # the longest machine
seq = train[train["sequence_id"] == seq_id].reset_index(drop=True)
full = make_features(seq)                                  # the whole diary

for c in [0, 10, 89, len(seq) // 2, len(seq) - 2]:         # several cut points
    cut = make_features(seq.iloc[:c + 1])                  # the diary cut after row c
    assert full.iloc[c].equals(cut.iloc[c]), c
print(f"No-future test passed on {seq_id} ({len(seq)} events): features at a cut do not change when later events are removed")

No-future test passed on TR00160 (250 events): features at a cut do not change when later events are removed


In [ ]:
import lightgbm as lgb
import pandas as pd
from eventflow.validation import evaluate

tr, va = split(examples, TUNING_FOLD)
tr_small = tr.sample(150_000, random_state=2210)          # smaller, so it runs in a few minutes
cat_type = pd.CategoricalDtype(categories=range(-1, 14))
X_tr = X.loc[tr_small["last_row"]].astype({c: cat_type for c in CATEGORICAL}) #select the last feautre row of every training question then convert 3 ev_lag columns into the category type
X_va = X.loc[va["last_row"]].astype({c: cat_type for c in CATEGORICAL}) #select the last feature row of every validation question then convert 3 ev_lag columns into the category type

model = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, verbose=-1)
model.fit(X_tr, tr_small["target"], eval_set=[(X_va, va["target"])],
          callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)])
assert len(model.classes_) == 14
print(evaluate(va, model.predict_proba(X_va)))
# logloss       = score on ALL validation cuts (main score)  (1 to 90,91,92,... 199)      
# logloss_mimic = score on only the validation cuts made like test.csv (eg 1 to 170 only)
# groups 1–2 (49 cols): logloss 1.544, mimic 1.541   (beats last-3 table 1.655)
# groups 1–3 (63 cols): logloss 1.518, mimic 1.510   → group 3 helps (−0.026)
# groups 1–4 (79 cols): logloss 1.517, mimic 1.507   → group 4 adds ~nothing (−0.0015, likely noise)

c:\Users\nampa\Documents\eventflow\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 30 rounds
[50]	valid_0's multi_logloss: 1.53301
[100]	valid_0's multi_logloss: 1.51706
Early stopping, best iteration is:
[105]	valid_0's multi_logloss: 1.51696
{'logloss': 1.5169588679501511, 'logloss_mimic': 1.5074436903925181, 'n_examples': 117311, 'n_sequences': 1300}


Quick check (same 150k sample, fold 0): groups 1–2 → 1.544; groups 1–3 → 1.518 (−0.026). Group 3 (events since) helps.

In [6]:
#hand check for since each event happend
import numpy as np

row = int(examples["last_row"].iloc[1234])
seq_id = train.loc[row, "sequence_id"]
seq_start = train.index[train["sequence_id"] == seq_id][0]
history = train.loc[seq_start:row, "event_type"].tolist()

def events_ago(history, event):
    """Slow, obvious version: walk backwards until we find the event."""
    for steps, e in enumerate(reversed(history)):
        if e == event:
            return steps
    return np.nan                                   # never happened in this sequence

for event in EVENT_CLASSES:
    fast, slow = X.loc[row, f"since_{event}"], events_ago(history, event)
    assert (np.isnan(fast) and np.isnan(slow)) or fast == slow, (event, fast, slow)
print("Group 3 hand check passed:", {e: events_ago(history, e) for e in EVENT_CLASSES[:4]})

Group 3 hand check passed: {'COMPONENT_REPLACED': 10, 'CONTROL_ALERT': nan, 'COOLING_ACTIVATED': 0, 'COOLING_FAULT': 82}


In [ ]:
#hand check for group 4 share event so far and mean severity so far
row = int(examples["last_row"].iloc[1234])
seq_id = train.loc[row, "sequence_id"]
seq_start = train.index[train["sequence_id"] == seq_id][0]
history = train.loc[seq_start:row, "event_type"].tolist()
sev_history = train.loc[seq_start:row, "severity"].tolist()

by_hand = {
    "share_RECOVERY": history.count("RECOVERY") / len(history),
    "share_TEMP_WARNING": history.count("TEMP_WARNING") / len(history),
    "mean_severity_so_far": sum(sev_history) / len(sev_history),
    "events_so_far": len(history),
}
for name, value in by_hand.items():
    assert np.isclose(X.loc[row, name], value), (name, X.loc[row, name], value)
print("Group 4 hand check passed:", {k: round(v, 3) for k, v in by_hand.items()})

Group 4 hand check passed: {'share_RECOVERY': 0.371, 'share_TEMP_WARNING': 0.124, 'mean_severity_so_far': 0.619, 'events_so_far': 97}
